# Week 6 Assignment

# Spark Architecture and Efficient Data Processing using PySpark

# Objective

The objective of this assignment is to understand the basic working of Apache Spark and perform common data processing tasks using PySpark. In this notebook, I will read a CSV dataset, inspect its structure, perform data cleaning, apply transformations and filters, create new columns, and save the processed data in different formats. Along the way, I will also understand important Spark concepts such as Lazy Evaluation, DAG, Spark Architecture, and the difference between CSV and Parquet.

## Dataset Information

For this assignment, no dataset was provided. Therefore, I created a sample retail dataset containing product details such as product ID, category, price, base price, order status, user ID, region, and priority.

The dataset is designed to demonstrate different Spark operations including filtering, schema handling, null value processing, transformations, and exporting processed data into different file formats.

# Step 1 : Import Required Libraries

Before starting any data processing task, the required PySpark libraries need to be imported. These libraries provide functions to create a Spark Session, work with DataFrames, perform data transformations, handle missing values, and manipulate columns.

In this assignment, I will mainly use the DataFrame API provided by PySpark because it is simple to read, optimized for performance, and widely used in data engineering projects.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

### Observation

The required libraries were imported successfully. These libraries will be used throughout the notebook to perform data loading, transformation, filtering, and data export operations.

# Step 2: Create Spark Session

A Spark Session is the starting point of every PySpark application. It allows us to interact with Spark and provides access to DataFrames, SQL functions, and other Spark features.

In this step, I am creating a Spark Session named **"Week6_Spark_Assignment"**, which will be used for all the operations performed in this notebook.

In [2]:
spark = SparkSession.builder \
    .appName("Week6_Spark_Assignment") \
    .getOrCreate()

In [3]:
spark

**Observation** 

The Spark Session was created successfully.This confirms that the PySpark environment is ready, and the remaining data processing tasks can now be performed using this session.

# Step 3 : Read CSV Dataset

The dataset is loaded from the data folder using the CSV reader. The header option is enabled so that Spark reads the first row as column names, and inferSchema is enabled so that Spark automatically detects the data types.

In [4]:
df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("../data/source.csv")

**Observation**

The CSV dataset was loaded successfully into a Spark DataFrame.

# Step 4 : Display the Dataset

Before performing any transformations, it is useful to inspect the data. Displaying a few records helps verify that the dataset has been loaded correctly.

In [5]:
df.show(5,truncate=False)

+----------+--------------+-----------+--------+----------+---------+------+-------+------+--------+
|product_id|old_name      |category   |price   |base_price|status   |amount|user_id|region|priority|
+----------+--------------+-----------+--------+----------+---------+------+-------+------+--------+
|P101      |Laptop        |Electronics|54585.08|49398     |Completed|3316  |NULL   |North |Medium  |
|P102      |Router        |Electronics|30054.4 |28451     |Completed|1252  |U1002  |North |Medium  |
|P103      |Jacket        |Fashion    |9178.16 |7859      |Pending  |326   |U1003  |South |Medium  |
|P104      |Microwave     |Appliances |24461.65|22379     |Pending  |718   |U1004  |North |High    |
|P105      |Water Purifier|Appliances |44424.19|36671     |Pending  |2496  |U1005  |North |Low     |
+----------+--------------+-----------+--------+----------+---------+------+-------+------+--------+
only showing top 5 rows


**Observation**

The first five records were displayed successfully. The dataset contains product information, prices, categories, regions, order status, and user details.

# Step 5 : Explore the Dataset

The total number of rows, columns, and column names are checked to understand the structure of the dataset before applying any transformations.

In [6]:
print("Rows :",df.count())
print("Columns :",len(df.columns))
print(df.columns)

Rows : 75
Columns : 10
['product_id', 'old_name', 'category', 'price', 'base_price', 'status', 'amount', 'user_id', 'region', 'priority']


**Observation**

The dataset contains all the expected columns and records. This confirms that the dataset has been loaded correctly.

# Step 6 : Check the Schema

Schema represents the structure of the DataFrame, including column names and their data types. Understanding the schema is important before performing any calculations or transformations.

In [7]:
df.printSchema()

root
 |-- product_id: string (nullable = true)
 |-- old_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: double (nullable = true)
 |-- base_price: integer (nullable = true)
 |-- status: string (nullable = true)
 |-- amount: integer (nullable = true)
 |-- user_id: string (nullable = true)
 |-- region: string (nullable = true)
 |-- priority: string (nullable = true)



**Observation**

Spark automatically identified the appropriate data types for each column using inferSchema=True. This reduces manual effort and allows numerical operations to be performed directly.

# Step 7 : Check Missing Values

Before performing data analysis, it is important to identify missing values. Missing values can affect filtering, calculations, and overall data quality.

In [8]:
from pyspark.sql.functions import col,isnan,when,count
df.select(
    [
    count(
    when(col(c).isNull(),c)
    ).alias(c)
    for c in df.columns
    ]
).show()

+----------+--------+--------+-----+----------+------+------+-------+------+--------+
|product_id|old_name|category|price|base_price|status|amount|user_id|region|priority|
+----------+--------+--------+-----+----------+------+------+-------+------+--------+
|         0|       0|       0|    0|         0|     0|     0|      7|     0|       0|
+----------+--------+--------+-----+----------+------+------+-------+------+--------+



**Observation**

The dataset contains a few missing values in the user_id column. These missing values will be handled in the next step before further processing.

# Step 8 : Handle Missing Values

After identifying the missing values, the next step is to handle them. In this dataset, the `user_id` column contains a few null values. For this assignment, I am removing rows where `user_id` is missing to ensure that the processed dataset contains only complete records.

In [9]:
df_clean = df.na.drop(subset=["user_id"])

In [10]:
print("Rows before cleaning :", df.count())

print("Rows after cleaning :", df_clean.count())

Rows before cleaning : 75
Rows after cleaning : 68


**Observation**

Rows containing null values in the `user_id` column were removed successfully. The cleaned DataFrame will be used for all further transformations and analysis.

# Step 9 : Select Required Columns

Selecting only the required columns helps reduce unnecessary data processing and improves readability. Here, I selected the `product_id` and `price` columns for products belonging to the **Electronics** category.

In [11]:
electronics_df = df_clean.filter(
    col("category")=="Electronics"
).select(
    "product_id",
    "price"
)

electronics_df.show()

+----------+--------+
|product_id|   price|
+----------+--------+
|      P102| 30054.4|
|      P107| 18954.6|
|      P110|57740.75|
|      P115| 51535.1|
|      P122|40663.55|
|      P128|73855.86|
|      P131|44765.16|
|      P132| 6184.05|
|      P135|36206.26|
|      P138|40060.86|
|      P140|27286.51|
|      P151|68238.03|
|      P157|34313.28|
|      P163|11712.36|
|      P165|29468.75|
|      P168|48748.62|
|      P171| 4128.51|
+----------+--------+



**Observation**

Only the required columns were displayed for products in the Electronics category. Selecting only the necessary columns reduces memory usage and improves processing efficiency.

# Step 10 : Filter Completed Orders

Filtering is one of the most commonly used DataFrame operations. In this step, I filtered the dataset to display only those orders where the status is **Completed** and the amount is greater than **1000**.

In [12]:
completed_orders = df_clean.filter(
    (col("status")=="Completed")&
    (col("amount")>1000)
)
completed_orders.show(truncate=False)

+----------+---------------+-----------+--------+----------+---------+------+-------+------+--------+
|product_id|old_name       |category   |price   |base_price|status   |amount|user_id|region|priority|
+----------+---------------+-----------+--------+----------+---------+------+-------+------+--------+
|P102      |Router         |Electronics|30054.4 |28451     |Completed|1252  |U1002  |North |Medium  |
|P108      |Smart Watch    |Fashion    |3556.45 |3164      |Completed|1393  |U1008  |North |Medium  |
|P109      |Bookshelf      |Furniture  |13947.05|12208     |Completed|2921  |U1009  |South |Medium  |
|P110      |Mouse          |Electronics|57740.75|54657     |Completed|1396  |U1010  |North |High    |
|P116      |Mixer          |Appliances |46011.35|40469     |Completed|3257  |U1016  |East  |Low     |
|P118      |Chair          |Furniture  |45431.6 |40752     |Completed|1786  |U1018  |East  |High    |
|P121      |Smart Watch    |Fashion    |9442.12 |7677      |Completed|1315  |U1021

**Observation**

The filtered DataFrame contains only completed orders with an amount greater than 1000. This demonstrates how multiple conditions can be combined using logical operators in Spark.

# Step 11 : Filter Data Using OR Condition

In this step, I filtered the dataset to display records where the region is **North** or the priority is **High**. This demonstrates the use of the logical OR operator while filtering a DataFrame.

In [13]:
north_high = df_clean.filter(
    (col("region")=="North") |
    (col("priority")=="High")
)
north_high.show(truncate=False)

+----------+--------------+-----------+--------+----------+---------+------+-------+------+--------+
|product_id|old_name      |category   |price   |base_price|status   |amount|user_id|region|priority|
+----------+--------------+-----------+--------+----------+---------+------+-------+------+--------+
|P102      |Router        |Electronics|30054.4 |28451     |Completed|1252  |U1002  |North |Medium  |
|P104      |Microwave     |Appliances |24461.65|22379     |Pending  |718   |U1004  |North |High    |
|P105      |Water Purifier|Appliances |44424.19|36671     |Pending  |2496  |U1005  |North |Low     |
|P107      |SSD           |Electronics|18954.6 |15735     |Cancelled|1253  |U1007  |North |High    |
|P108      |Smart Watch   |Fashion    |3556.45 |3164      |Completed|1393  |U1008  |North |Medium  |
|P110      |Mouse         |Electronics|57740.75|54657     |Completed|1396  |U1010  |North |High    |
|P111      |Refrigerator  |Appliances |76142.64|67435     |Cancelled|2179  |U1011  |South |

**Observation**

The output contains all records that satisfy either of the given conditions. Spark evaluates both conditions and returns the matching rows efficiently.

# Step 12 : Rename a Column

Sometimes column names need to be updated to make them more meaningful or to follow naming conventions. In this step, I renamed the column `old_name` to `new_name`.

In [14]:
df_clean = df_clean.withColumnRenamed(
    "old_name",
    "new_name"
)
df_clean.show(5)

+----------+--------------+-----------+--------+----------+---------+------+-------+------+--------+
|product_id|      new_name|   category|   price|base_price|   status|amount|user_id|region|priority|
+----------+--------------+-----------+--------+----------+---------+------+-------+------+--------+
|      P102|        Router|Electronics| 30054.4|     28451|Completed|  1252|  U1002| North|  Medium|
|      P103|        Jacket|    Fashion| 9178.16|      7859|  Pending|   326|  U1003| South|  Medium|
|      P104|     Microwave| Appliances|24461.65|     22379|  Pending|   718|  U1004| North|    High|
|      P105|Water Purifier| Appliances|44424.19|     36671|  Pending|  2496|  U1005| North|     Low|
|      P106|         Watch|    Fashion|10580.82|      9544|  Pending|  1781|  U1006| South|  Medium|
+----------+--------------+-----------+--------+----------+---------+------+-------+------+--------+
only showing top 5 rows


**Observation**

The column was renamed successfully. Renaming columns improves readability without affecting the data stored in the DataFrame.

# Step 13 : Change Data Type

The `price` column is converted to the **Double** data type. Converting numeric columns to the appropriate data type is useful for performing mathematical calculations accurately.

In [15]:
df_clean = df_clean.withColumn(
    "price",
    col("price").cast("double")
)
df_clean.printSchema()

root
 |-- product_id: string (nullable = true)
 |-- new_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: double (nullable = true)
 |-- base_price: integer (nullable = true)
 |-- status: string (nullable = true)
 |-- amount: integer (nullable = true)
 |-- user_id: string (nullable = true)
 |-- region: string (nullable = true)
 |-- priority: string (nullable = true)



**Observation**

The `price` column has been successfully converted to the Double data type, making it suitable for numerical calculations and aggregations.

# Step 14 : Create a New Column

A new column named `final_price` is created by adding 18% tax to the `base_price` column. This demonstrates how Spark can generate new columns using existing data.

In [16]:
df_clean = df_clean.withColumn(
    "final_price",
    round(col("base_price")*1.18,2)
)

df_clean.select(
    "base_price",
    "final_price"
).show(10)

+----------+-----------+
|base_price|final_price|
+----------+-----------+
|     28451|   33572.18|
|      7859|    9273.62|
|     22379|   26407.22|
|     36671|   43271.78|
|      9544|   11261.92|
|     15735|    18567.3|
|      3164|    3733.52|
|     12208|   14405.44|
|     54657|   64495.26|
|     67435|    79573.3|
+----------+-----------+
only showing top 10 rows


**Observation**

The new column `final_price` was created successfully using the existing `base_price` values. Spark performed the calculation efficiently across all records.

# Step 15 : Understanding Transformations and Actions

Throughout this notebook, different DataFrame operations have been performed.

**Transformations** create a new DataFrame but do not execute immediately. Examples used in this notebook include:
- `filter()`
- `select()`
- `withColumn()`
- `withColumnRenamed()`

An **Action** triggers Spark to execute all pending transformations. Examples used in this notebook include:
- `show()`
- `count()`
- `printSchema()`

Spark combines multiple transformations and executes them only when an action is called. This behavior improves performance and is known as **Lazy Evaluation**.

# Step 16 : Save the Processed Dataset in CSV Format

After cleaning and transforming the dataset, the next step is to save the processed data. Saving the output allows the transformed dataset to be reused for reporting or further analysis.

Here, the cleaned DataFrame is written into the **processed_csv** folder in CSV format.

In [17]:
import os

os.makedirs("../output/processed_csv", exist_ok=True)
os.makedirs("../output/processed_parquet", exist_ok=True)

In [18]:
df_clean.write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("../output/processed_csv")

**Observation**

The processed dataset was successfully saved in CSV format. Spark created the output files inside the `processed_csv` folder.

# Step 17 : Save the Processed Dataset as Parquet

In addition to CSV, Spark also supports the Parquet file format. Parquet stores data in a columnar format, making it more efficient for storage and analytical processing. It also provides better compression and faster query performance compared to CSV.

In this step, the processed dataset is saved in Parquet format.

In [19]:
df_clean.write \
    .mode("overwrite") \
    .parquet("../output/processed_parquet")

**Observation**

The processed dataset was successfully saved in Parquet format. Spark stored the data in a compressed columnar format, making it suitable for efficient analytical queries.

# Step 18 : Read the Saved Parquet Dataset

To verify that the Parquet dataset has been written successfully, it is loaded again into a new Spark DataFrame.Reading the file confirms that the output can be reused for further processing.

In [20]:
parquet_df = spark.read.parquet("../output/processed_parquet")

In [21]:
parquet_df.show(5, truncate=False)

+----------+--------------+-----------+--------+----------+---------+------+-------+------+--------+-----------+
|product_id|new_name      |category   |price   |base_price|status   |amount|user_id|region|priority|final_price|
+----------+--------------+-----------+--------+----------+---------+------+-------+------+--------+-----------+
|P102      |Router        |Electronics|30054.4 |28451     |Completed|1252  |U1002  |North |Medium  |33572.18   |
|P103      |Jacket        |Fashion    |9178.16 |7859      |Pending  |326   |U1003  |South |Medium  |9273.62    |
|P104      |Microwave     |Appliances |24461.65|22379     |Pending  |718   |U1004  |North |High    |26407.22   |
|P105      |Water Purifier|Appliances |44424.19|36671     |Pending  |2496  |U1005  |North |Low     |43271.78   |
|P106      |Watch         |Fashion    |10580.82|9544      |Pending  |1781  |U1006  |South |Medium  |11261.92   |
+----------+--------------+-----------+--------+----------+---------+------+-------+------+-----

**Observation**

The Parquet file was loaded successfully. The displayed records confirm that the dataset was saved correctly and can be reused in future Spark applications.

# Step 19 : Building a Simple ETL Pipeline

The workflow implemented in this notebook follows the ETL (Extract, Transform, Load) process commonly used in data engineering.

- **Extract:** Read the source dataset from a CSV file.
- **Transform:** Handle missing values, filter records, rename columns, change data types, and create a new column.
- **Load:** Save the processed dataset into CSV and Parquet formats.

This demonstrates a simple Spark-based ETL pipeline.

In [22]:
print("Extract : CSV file loaded")
print("Transform : Cleaning and transformations completed")
print("Load : Data saved in CSV and Parquet formats")
print("ETL Pipeline Completed Successfully")

Extract : CSV file loaded
Transform : Cleaning and transformations completed
Load : Data saved in CSV and Parquet formats
ETL Pipeline Completed Successfully


**Observation**

The ETL pipeline was completed successfully. The dataset was extracted, transformed according to the requirements, and finally loaded into the desired output formats.

# Step 20 : Comparing CSV and Parquet

During this assignment, the processed dataset was stored in both CSV and Parquet formats.

| CSV | Parquet |
|------|----------|
| Row-based storage | Column-based storage |
| Human-readable | Binary format |
| Larger file size | Better compression |
| Slower query performance | Faster analytical queries |
| No Schema support | Stores schema information |

Parquet is generally preferred in Spark because it improves storage efficiency and query performance, especially for large datasets.

**Observation**

Both formats store the same data, but Parquet is more efficient due to its columnar storage and compression capabilities.

# Step 21 : Understanding Predicate Pushdown

Predicate Pushdown is an optimization technique used by Spark when working with columnar file formats such as Parquet. Instead of loading the entire dataset into memory, Spark pushes the filtering condition to the storage layer. As a result, only the required rows and columns are read from the disk.

This optimization reduces disk I/O, decreases memory usage, and improves query execution time, especially when working with very large datasets.

In [23]:
electronics_parquet = spark.read.parquet("../output/processed_parquet") \
    .filter(col("category") == "Electronics")
    
electronics_parquet.show(5)

+----------+--------+-----------+--------+----------+---------+------+-------+------+--------+-----------+
|product_id|new_name|   category|   price|base_price|   status|amount|user_id|region|priority|final_price|
+----------+--------+-----------+--------+----------+---------+------+-------+------+--------+-----------+
|      P102|  Router|Electronics| 30054.4|     28451|Completed|  1252|  U1002| North|  Medium|   33572.18|
|      P107|     SSD|Electronics| 18954.6|     15735|Cancelled|  1253|  U1007| North|    High|    18567.3|
|      P110|   Mouse|Electronics|57740.75|     54657|Completed|  1396|  U1010| North|    High|   64495.26|
|      P115|     SSD|Electronics| 51535.1|     48033|  Pending|  3375|  U1015|  East|     Low|   56678.94|
|      P122|  Router|Electronics|40663.55|     37101|Completed|   590|  U1022| North|    High|   43779.18|
+----------+--------+-----------+--------+----------+---------+------+-------+------+--------+-----------+
only showing top 5 rows


**Observation**

The required records were filtered successfully from the Parquet dataset. In large-scale datasets, Spark reads only the necessary data blocks, making Parquet much more efficient than CSV.

# Step 22 : Understanding Lazy Evaluation and DAG

Spark follows the concept of Lazy Evaluation. Transformations such as `filter()`, `select()`, and `withColumn()` are not executed immediately. Instead, Spark records all the transformations and builds a Directed Acyclic Graph (DAG).

Execution begins only when an Action such as `show()`, `count()`, or `write()` is called. Spark then analyzes the DAG and creates an optimized execution plan, reducing unnecessary computations and improving performance.

In [24]:
lazy_df = df_clean.filter(col("price") > 500) \
    .select("product_id", "category", "price")
print("Transformations created successfully.")
print("No execution has happened yet.")

Transformations created successfully.
No execution has happened yet.


In [25]:
lazy_df.show()

+----------+-----------+--------+
|product_id|   category|   price|
+----------+-----------+--------+
|      P102|Electronics| 30054.4|
|      P103|    Fashion| 9178.16|
|      P104| Appliances|24461.65|
|      P105| Appliances|44424.19|
|      P106|    Fashion|10580.82|
|      P107|Electronics| 18954.6|
|      P108|    Fashion| 3556.45|
|      P109|  Furniture|13947.05|
|      P110|Electronics|57740.75|
|      P111| Appliances|76142.64|
|      P113|  Furniture| 8957.02|
|      P114|    Fashion| 7663.51|
|      P115|Electronics| 51535.1|
|      P116| Appliances|46011.35|
|      P117|  Furniture|47138.48|
|      P118|  Furniture| 45431.6|
|      P119|  Furniture| 8509.86|
|      P120|    Fashion|  5595.1|
|      P121|    Fashion| 9442.12|
|      P122|Electronics|40663.55|
+----------+-----------+--------+
only showing top 20 rows


**Observation**

The transformations were executed only when the `show()` action was called. This demonstrates Spark's Lazy Evaluation strategy and how it optimizes execution using a DAG.

# Step 23 : Understanding Wide Transformations

Some Spark transformations require data to move between different partitions. These are known as Wide Transformations and involve a process called Shuffle.

Operations such as `groupBy()`, `join()`, and `orderBy()` require Spark to redistribute data across executors before producing the final result. Although powerful, shuffle operations are comparatively expensive and should be minimized whenever possible.

In [26]:
category_summary = df_clean.groupBy("category") \
                           .agg(avg("price").alias("Average_Price"))

category_summary.show()

+-----------+------------------+
|   category|     Average_Price|
+-----------+------------------+
|    Fashion| 6673.782173913044|
|Electronics| 36700.97941176471|
|  Furniture|          27895.61|
| Appliances|45460.988333333335|
+-----------+------------------+



**Observation**

The average price for each category was calculated successfully. Since `groupBy()` redistributes data across partitions, this operation performs a shuffle internally.

# Step 24 : Spark Architecture and Execution Modes

Apache Spark follows a distributed architecture consisting of three major components:

- **Driver Program:** Creates the Spark Session, plans the execution, and coordinates the application.
- **Cluster Manager:** Allocates resources and manages worker nodes.
- **Executors:** Execute tasks, process partitions of data, and return the results to the Driver.

Spark supports two execution modes:

- **Client Mode:** The Driver runs on the local machine, while Executors run on worker nodes. This mode is generally used for development and testing.
- **Cluster Mode:** Both the Driver and Executors run inside the cluster. This mode is preferred for production workloads because it provides better scalability and fault tolerance.

In [27]:
print("Spark Application Name :", spark.sparkContext.appName)
print("Spark Version :", spark.version)
print("Master :", spark.sparkContext.master)

Spark Application Name : Week6_Spark_Assignment
Spark Version : 4.1.2
Master : local[*]


**Observation**

The Spark application is running in Local Mode (local[*]), which is suitable for development and testing.

# Step 25 : Best Practices While Processing Large Datasets

When working with very large datasets, it is important to follow efficient Spark practices.

- Use `show()` to inspect a small number of records.
- Avoid using `collect()` on very large datasets because it transfers all records to the Driver's memory.
- Prefer columnar formats such as Parquet.
- Minimize shuffle operations whenever possible.
- Select only the required columns instead of reading the complete dataset.

In [28]:
df_clean.show(5)

+----------+--------------+-----------+--------+----------+---------+------+-------+------+--------+-----------+
|product_id|      new_name|   category|   price|base_price|   status|amount|user_id|region|priority|final_price|
+----------+--------------+-----------+--------+----------+---------+------+-------+------+--------+-----------+
|      P102|        Router|Electronics| 30054.4|     28451|Completed|  1252|  U1002| North|  Medium|   33572.18|
|      P103|        Jacket|    Fashion| 9178.16|      7859|  Pending|   326|  U1003| South|  Medium|    9273.62|
|      P104|     Microwave| Appliances|24461.65|     22379|  Pending|   718|  U1004| North|    High|   26407.22|
|      P105|Water Purifier| Appliances|44424.19|     36671|  Pending|  2496|  U1005| North|     Low|   43271.78|
|      P106|         Watch|    Fashion|10580.82|      9544|  Pending|  1781|  U1006| South|  Medium|   11261.92|
+----------+--------------+-----------+--------+----------+---------+------+-------+------+-----

**Observation**
Only the first five records were displayed. Using `show()` is a safer approach for exploring large datasets because it avoids loading the entire dataset into the Driver's memory.

# Conclusion

In this assignment, I explored the fundamentals of Apache Spark and implemented a complete data processing workflow using PySpark. I learned about Spark Architecture, execution modes, Lazy Evaluation, DAG, DataFrame transformations, filtering, schema handling, null value processing, wide transformations, and Predicate Pushdown.

I also built a simple ETL pipeline by reading data from a CSV file, transforming it using various DataFrame operations, and saving the processed output in both CSV and Parquet formats. Finally, I compared different storage formats and understood the importance of following best practices while processing large datasets using Apache Spark.

This assignment provided hands-on experience with Spark DataFrame operations and demonstrated how Spark efficiently processes large datasets using distributed computing concepts.